# AgriSense - Visualizing Trends and Relationships

Welcome to Sections 4.41 and 4.42! In this notebook, we will continue our data visualization journey by learning how to create **Line Plots** and **Scatter Plots**.

### Why are we doing this?
1. **Line Plots (Trends over Time):** To see how crop prices change month by month. This helps us verify if the data makes sense before we build the Market Dashboard for the frontend.
2. **Scatter Plots (Relationships):** To understand how two variables relate to each other (e.g., "Does more rainfall mean higher yield?"). Understanding these relationships tells us exactly which features will be most useful for our Machine Learning models later!

Let's dive in!

In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Ignore warnings for a cleaner notebook output
warnings.filterwarnings('ignore')

# Create the outputs folder if it doesn't exist to save our plots
Path("../outputs/figures").mkdir(parents=True, exist_ok=True)

# Set a professional visual style for all plots
sns.set_theme(style="whitegrid")
print("Libraries imported successfully! Ready to plot.")

## Section 4.41: Line Plots - Tracking Prices Over Time

First, we need to load our price data and make sure the date column is actually treated as a "Date" by Python (using `pd.to_datetime()`).

In [ ]:
# 2. Load the dataset
data_path = Path("../data/raw/mandi_prices.csv")
df = pd.read_csv(data_path)

# Standardize column names (from previous lessons)
df.columns = df.columns.str.lower().str.strip()

# CRITICAL STEP: Convert the date column from plain text to proper datetime objects
df['date'] = pd.to_datetime(df['date'])

# Sort the data chronologically so our line plots draw correctly from left to right!
df = df.sort_values('date')

print(f"Data loaded and dates converted! Date range: {df['date'].min().date()} to {df['date'].max().date()}")
display(df.head(3))

### Plotting a Single Crop's Trend
Let's see how the price of **Wheat** changed over time.

In [ ]:
# 3. Filter data just for Wheat
wheat_df = df[df['commodity'].str.lower() == 'wheat']

plt.figure(figsize=(10, 5))

# Create the line plot
sns.lineplot(data=wheat_df, x='date', y='modal_price', marker='o', linewidth=2, color='darkorange')

plt.title('Wheat Price Trend Over Time', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Modal Price (₹)', fontsize=12)

# Save and show
plt.savefig('../outputs/figures/wheat_trend.png', dpi=300, bbox_inches='tight')
plt.show()

**💡 Insight:** 
By simply looking at the line, you can tell if Wheat prices are stable, rising, or falling. A flat, stable line means normal market conditions, whereas sharp spikes might indicate a supply shortage.

### Comparing Multiple Crops
What if we want to compare Wheat, Rice, and Tomato all on the same graph? Seaborn makes this incredibly easy using the `hue` parameter.

In [ ]:
# 4. Filter data for a few specific crops to compare
crops_to_compare = ['Wheat', 'Rice', 'Tomato']
comparison_df = df[df['commodity'].isin(crops_to_compare)]

plt.figure(figsize=(12, 6))

# Plot multiple lines by setting hue='commodity'
sns.lineplot(data=comparison_df, x='date', y='modal_price', hue='commodity', marker='o', linewidth=2)

plt.title('Price Trends: Wheat vs Rice vs Tomato', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Modal Price (₹)', fontsize=12)
plt.legend(title='Crop')

plt.savefig('../outputs/figures/multicrop_trend.png', dpi=300, bbox_inches='tight')
plt.show()

**💡 Insight:** 
* Notice how Rice is consistently priced higher than Wheat.
* Tomatoes might show more volatility (ups and downs) because they are highly perishable compared to grains.
* This exact chart design is what you will recreate using Recharts/Chart.js on the Frontend Dashboard!

## Section 4.42: Scatter Plots - Exploring Relationships

To build a Yield Prediction model later, we need to know what affects yield. Let's simulate some environmental data (Rainfall, Temperature) since our raw prices dataset doesn't have it natively, and then explore how they correlate with Yield using Scatter Plots.

In [ ]:
# 5. Generate mock environmental and yield data for educational purposes
np.random.seed(42)

# Create a new dataframe for environmental analysis
env_df = pd.DataFrame()

# Simulate Rainfall (mm) - 50 to 200 mm
env_df['rainfall_mm'] = np.random.uniform(50, 200, 100)

# Simulate Temperature (C) - 20 to 40 degrees
env_df['temperature_c'] = np.random.uniform(20, 40, 100)

# Yield formula: base yield + bonus from rain - penalty from extreme heat + random noise
# (This creates a realistic looking relationship to plot)
env_df['yield_qha'] = 10 + (env_df['rainfall_mm'] * 0.15) - (env_df['temperature_c'] * 0.2) + np.random.normal(0, 3, 100)

display(env_df.head())

### Scatter Plot 1: Rainfall vs. Yield
Does more rain generally lead to higher crop yield? Let's use `sns.regplot` which plots the dots AND draws a "line of best fit" (trend line) automatically!

In [ ]:
# 6. Scatter plot with regression line for Rainfall vs Yield
plt.figure(figsize=(9, 6))

sns.regplot(data=env_df, x='rainfall_mm', y='yield_qha', 
            scatter_kws={'alpha':0.6, 'color':'blue'}, 
            line_kws={'color':'red', 'linewidth':2})

plt.title('Relationship: Rainfall vs. Crop Yield', fontsize=16)
plt.xlabel('Rainfall (mm)', fontsize=12)
plt.ylabel('Yield (Quintals/Hectare)', fontsize=12)

plt.savefig('../outputs/figures/rain_vs_yield.png', dpi=300, bbox_inches='tight')
plt.show()

**💡 Insight:** 
* The red trend line goes **up** from left to right. This indicates a **positive correlation**.
* Higher rainfall generally leads to higher yield in this dataset.
* Because this relationship is clear, "Rainfall" will be an excellent feature for our Machine Learning model!

### Scatter Plot 2: Temperature vs. Yield
What happens when it gets too hot?

In [ ]:
# 7. Scatter plot with regression line for Temperature vs Yield
plt.figure(figsize=(9, 6))

sns.regplot(data=env_df, x='temperature_c', y='yield_qha', 
            scatter_kws={'alpha':0.6, 'color':'orange'}, 
            line_kws={'color':'red', 'linewidth':2})

plt.title('Relationship: Temperature vs. Crop Yield', fontsize=16)
plt.xlabel('Temperature (°C)', fontsize=12)
plt.ylabel('Yield (Quintals/Hectare)', fontsize=12)

plt.savefig('../outputs/figures/temp_vs_yield.png', dpi=300, bbox_inches='tight')
plt.show()

**💡 Insight:** 
* The trend line goes **down** from left to right. This is a **negative correlation**.
* As temperatures approach 40°C, the crop yield tends to drop due to heat stress.
* Temperature is another highly valuable feature for predictive modeling.

## Conclusion &amp; Next Steps

### Key Insights from Visualizations:
1. **Time-Series Confidence:** By plotting lines (Section 4.41), we proved our date sorting and data integrity are solid. We now know that our FastAPI backend can safely send this data to the React dashboard to render beautiful historical price charts.
2. **Feature Importance Validation:** By plotting scatter graphs and trend lines (Section 4.42), we visually confirmed that Rainfall and Temperature have a mathematical relationship with Yield. This is *proof* that feeding these columns to a Machine Learning algorithm will give us accurate predictions.

### What's Next?
In the next section, we will move away from visualizations and start preparing our data mathematically for Machine Learning models by performing **Data Transformation and Feature Engineering**. 

You're doing great! 🌾📈